[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/06-multi-location-comparison.ipynb)

# Multi-Location Comparison

One of the most powerful applications of spatial demographic analysis is **comparison**.
A single location's census profile is informative, but placing two or three locations
side by side transforms raw numbers into actionable insight. Which city center has more
people within a 15-minute drive? Where is median income highest? Where is poverty
concentrated?

SocialMapper provides two functions that make this workflow trivial:

| Function | Purpose |
|---|---|
| `analyze_multiple_pois` | Run the full isochrone-to-census pipeline for a list of locations, aggregate per-location statistics, and optionally rank locations against each other. |
| `generate_report` | Render the analysis results as a styled HTML report suitable for sharing or embedding. |

In this notebook you will learn how to:

1. Understand **why** comparative spatial analysis matters
2. See **how** `analyze_multiple_pois` works under the hood
3. Analyze three NC cities (Raleigh, Charlotte, Asheville) at once
4. Inspect per-location aggregated statistics
5. Read and visualize comparison rankings
6. Create grouped bar charts for population and income
7. Compare walk vs. drive modes for the same location
8. Visualize all variables in a horizontal bar chart
9. Generate and display an HTML report
10. Use coordinate tuples as location inputs

## Setup

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper @ git+https://github.com/mihiarc/socialmapper.git'

from socialmapper import analyze_multiple_pois, generate_report
from IPython.display import HTML, display

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

---

## Why Compare Locations?

A single-location analysis tells you *what* the demographics look like around a point.
Comparative analysis tells you *how locations differ from each other* — and that is
where policy, planning, and investment decisions actually happen.

Consider three real-world scenarios:

**1. Spatial inequity detection.**
Two neighborhoods may have similar total populations, but vastly different access to
healthcare, education, or transit. Comparing isochrone-level demographics across
locations surfaces these disparities quantitatively rather than relying on intuition.

**2. Resource allocation.**
A nonprofit deciding where to open a new food pantry wants to know which candidate
site serves the most people living in poverty within a 15-minute drive. Comparative
analysis gives a direct, data-driven ranking.

**3. Policy evaluation.**
A state agency can compare median income and poverty rates across metro areas to
identify where economic development programs are most needed.

In every case the pattern is the same: define a set of locations, measure the same
demographic variables around each one using identical parameters (travel time, mode),
and then rank the results. That is exactly what `analyze_multiple_pois` automates.

---

## How `analyze_multiple_pois` Works

Under the hood, `analyze_multiple_pois` runs a four-stage pipeline:

```
locations ──┐
            │  (parallel, up to 4 threads)
            ▼
  ┌─────────────────────────┐
  │ For each location:      │
  │  1. create_isochrone()  │  ← Valhalla routing engine
  │  2. get_census_data()   │  ← ACS 5-year estimates
  │  3. Aggregate per-loc   │  ← total, mean, min, max, count
  └─────────────────────────┘
            │
            ▼
  ┌─────────────────────────┐
  │ 4. Compare & rank       │  ← only if compare=True
  └─────────────────────────┘
            │
            ▼
        results dict
```

**Stage 1 — Isochrone creation.** For each location, a Valhalla isochrone is generated
showing the area reachable within the given travel time and mode. Locations are processed
in parallel using a thread pool (up to 4 workers), so analyzing three cities takes roughly
the same wall-clock time as analyzing one.

**Stage 2 — Census data fetch.** The isochrone polygon is used to identify intersecting
census block groups via the TIGERweb API, then ACS 5-year estimates are pulled for the
requested variables.

**Stage 3 — Aggregation.** For each variable, block-group-level values are aggregated
into summary statistics:

| Statistic | Meaning |
|---|---|
| `total` | Sum across all block groups in the isochrone (e.g., total population reachable) |
| `mean` | Average value per block group (e.g., average median income per block group) |
| `min` | Lowest single block group value |
| `max` | Highest single block group value |
| `count` | Number of block groups with non-null data for that variable |

**Stage 4 — Comparison.** When `compare=True` (the default), the function ranks locations
by their `total` value for each variable, and identifies the `highest` and `lowest` location.

---

## 1. Three-City Analysis

Let us start with the core workflow: analyze three North Carolina cities with the same
travel time and mode, requesting two key demographic variables.

We will use:
- **Raleigh** — the state capital, part of the Research Triangle
- **Charlotte** — the largest city in NC, a major banking hub
- **Asheville** — a smaller mountain city in western NC

All three get a 15-minute driving isochrone, and we request `population` and
`median_income`.

In [ ]:
cities = ["Raleigh, NC", "Charlotte, NC", "Asheville, NC"]

results = analyze_multiple_pois(
    locations=cities,
    travel_time=15,
    travel_mode="drive",
    variables=["population", "median_income"],
)

print(f"Locations analyzed: {len(results['locations'])}")
print(f"Has comparison:     {'comparison' in results}")
print(f"Metadata:           {results['metadata']}")

The return value is a dictionary with three top-level keys:

- **`locations`** — a list of per-location result dicts (one per input location)
- **`comparison`** — a dict ranking locations for each variable (present when `compare=True`)
- **`metadata`** — the parameters you passed in (travel_time, travel_mode, variables)

Let us dig into each part.

---

## 2. Per-Location Results

Each entry in `results['locations']` is a dictionary describing one location's analysis.
The most important fields are:

| Field | Type | Description |
|---|---|---|
| `location` | str | The resolved location name |
| `isochrone` | dict | GeoJSON Feature of the travel-time polygon |
| `census_data` | dict | Raw `{geoid: {var: value}}` census data |
| `aggregated` | dict | Summary stats per variable: `{var: {total, mean, min, max, count}}` |
| `block_group_count` | int | Number of census block groups intersecting the isochrone |

The `aggregated` dictionary is where the action is. For a variable like `population`,
`total` means "total people living within the 15-minute drive", and `mean` means
"average population per block group". For `median_income`, `total` is less meaningful
(you would not sum median incomes), but `mean` gives you the average block-group-level
median income — a useful measure of area-wide economic conditions.

In [ ]:
for loc in results["locations"]:
    name = loc["location"]
    bg_count = loc["block_group_count"]
    pop = loc["aggregated"].get("population", {})
    inc = loc["aggregated"].get("median_income", {})
    print(f"\n--- {name} ---")
    print(f"  Block groups:       {bg_count}")
    print(f"  Total population:   {pop.get('total', 0):,.0f}")
    print(f"  Mean pop/BG:        {pop.get('mean', 0):,.0f}")
    print(f"  Pop range:          {pop.get('min', 0):,.0f} – {pop.get('max', 0):,.0f}")
    print(f"  Mean median income: ${inc.get('mean', 0):,.0f}")
    print(f"  Income range:       ${inc.get('min', 0):,.0f} – ${inc.get('max', 0):,.0f}")

Notice how the `min` and `max` fields reveal within-area variation. A city might have a
healthy average median income but a very low `min` value, indicating that at least one
block group within the 15-minute isochrone has significantly lower incomes than the rest.
This kind of intra-area inequality is invisible when you only look at totals.

---

## 3. Visualizing Population: Grouped Bar Chart

Numbers in a terminal are useful for scripting, but a chart makes the comparison
immediately legible. Let us build a grouped bar chart that shows total population
alongside block group count for each city.

In [ ]:
# Extract data for plotting
city_names = [loc["location"] for loc in results["locations"]]
total_pop = [loc["aggregated"]["population"]["total"] for loc in results["locations"]]
bg_counts = [loc["block_group_count"] for loc in results["locations"]]

fig, ax1 = plt.subplots(figsize=(9, 5))

x = np.arange(len(city_names))
bar_width = 0.4

bars1 = ax1.bar(x - bar_width / 2, total_pop, bar_width,
                label="Total Population", color="#4C78A8", edgecolor="white")
ax1.set_ylabel("Total Population", color="#4C78A8", fontsize=11)
ax1.tick_params(axis="y", labelcolor="#4C78A8")

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, height + 1000,
             f"{height:,.0f}", ha="center", va="bottom", fontsize=9, color="#4C78A8")

# Second y-axis for block group count
ax2 = ax1.twinx()
bars2 = ax2.bar(x + bar_width / 2, bg_counts, bar_width,
                label="Block Groups", color="#F58518", edgecolor="white")
ax2.set_ylabel("Block Group Count", color="#F58518", fontsize=11)
ax2.tick_params(axis="y", labelcolor="#F58518")

for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, height + 1,
             f"{height}", ha="center", va="bottom", fontsize=9, color="#F58518")

ax1.set_xticks(x)
ax1.set_xticklabels(city_names, fontsize=11)
ax1.set_title("Population within 15-Minute Drive", fontsize=13, fontweight="bold", pad=12)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", framealpha=0.9)

fig.tight_layout()
plt.show()

The blue bars show total population reachable within 15 minutes of driving from each
city center. The orange bars show how many census block groups fall within that
isochrone. Charlotte, as the largest city, will typically have the highest population.
Asheville, surrounded by mountains, will have fewer block groups and a smaller
reachable population.

This kind of chart is especially useful when presenting to non-technical stakeholders:
it immediately communicates the scale difference between locations.

---

## 4. Visualizing Income: Grouped Bar Chart

For income, the `mean` statistic is more meaningful than `total` (summing median incomes
across block groups does not produce a useful number). Let us compare the mean median
income for each city.

In [ ]:
mean_income = [loc["aggregated"]["median_income"]["mean"] for loc in results["locations"]]
min_income = [loc["aggregated"]["median_income"]["min"] for loc in results["locations"]]
max_income = [loc["aggregated"]["median_income"]["max"] for loc in results["locations"]]

fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(city_names))
bar_width = 0.5

bars = ax.bar(x, mean_income, bar_width, color="#59A14F", edgecolor="white",
              label="Mean Median Income")

# Error bars showing min/max range
error_low = [m - lo for m, lo in zip(mean_income, min_income)]
error_high = [hi - m for m, hi in zip(mean_income, max_income)]
ax.errorbar(x, mean_income, yerr=[error_low, error_high],
            fmt="none", ecolor="#333333", capsize=5, capthick=1.5, linewidth=1.5)

for bar, val in zip(bars, mean_income):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1500,
            f"${val:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(city_names, fontsize=11)
ax.set_ylabel("Median Income ($)", fontsize=11)
ax.set_title("Mean Median Income by Block Group (15-min Drive)",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(loc="upper right", framealpha=0.9)

# Add annotation explaining the error bars
ax.annotate("Error bars show min/max\nacross block groups",
            xy=(0.98, 0.02), xycoords="axes fraction",
            ha="right", va="bottom", fontsize=8, color="#666666",
            fontstyle="italic")

fig.tight_layout()
plt.show()

The green bars show the average median income across block groups within each city's
isochrone. The error bars (whiskers) extend from the lowest to the highest individual
block group, showing you the **spread** of income within the area.

A long error bar means the 15-minute drive radius contains both very wealthy and very
low-income block groups — a sign of economic diversity (or inequality, depending on
your perspective). A short error bar means the area is more economically homogeneous.

---

## 5. Comparison Rankings

When `compare=True` (the default), `analyze_multiple_pois` produces a `comparison`
dictionary that ranks locations for each requested variable.

The structure is:

```python
results["comparison"] = {
    "population": {
        "ranked": [
            {"location": "Charlotte, NC", "total": ..., "mean": ..., ...},
            {"location": "Raleigh, NC", ...},
            {"location": "Asheville, NC", ...},
        ],
        "highest": "Charlotte, NC",
        "lowest": "Asheville, NC",
    },
    "median_income": { ... },
}
```

Rankings are sorted by `total` in descending order. The `highest` and `lowest` fields
give you a quick answer without parsing the list.

In [ ]:
comparison = results["comparison"]

for var, info in comparison.items():
    print(f"\n{'=' * 50}")
    print(f"  Variable: {var}")
    print(f"  Highest:  {info['highest']}")
    print(f"  Lowest:   {info['lowest']}")
    print(f"{'=' * 50}")
    print(f"  {'Rank':<6} {'Location':<20} {'Total':>12} {'Mean':>10}")
    print(f"  {'-'*6} {'-'*20} {'-'*12} {'-'*10}")
    for rank_idx, rank in enumerate(info["ranked"], 1):
        print(f"  {rank_idx:<6} {rank['location']:<20} {rank['total']:>12,.0f} {rank['mean']:>10,.0f}")

---

## 6. Walk vs. Drive: Side-by-Side Comparison

A 15-minute walk covers far less ground than a 15-minute drive. This has profound
equity implications: people without cars have access to a much smaller pool of services,
jobs, and community resources.

Let us quantify this by running the same analysis for a single location (Raleigh) under
both travel modes, then visualizing the difference.

In [ ]:
mode_data = {}

for mode in ["drive", "walk"]:
    mode_results = analyze_multiple_pois(
        locations=["Raleigh, NC"],
        travel_time=15,
        travel_mode=mode,
        variables=["population", "median_income"],
        compare=False,
    )
    loc = mode_results["locations"][0]
    mode_data[mode] = {
        "population": loc["aggregated"]["population"]["total"],
        "median_income": loc["aggregated"]["median_income"]["mean"],
        "block_groups": loc["block_group_count"],
    }
    print(f"{mode:>5}: {mode_data[mode]['population']:>10,.0f} people "
          f"in {mode_data[mode]['block_groups']} block groups, "
          f"mean income ${mode_data[mode]['median_income']:,.0f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

modes = ["drive", "walk"]
colors = ["#4C78A8", "#E45756"]

# Population comparison
pop_vals = [mode_data[m]["population"] for m in modes]
bars1 = ax1.bar(modes, pop_vals, color=colors, edgecolor="white", width=0.5)
ax1.set_title("Total Population Reachable\n(Raleigh, 15 min)", fontsize=12, fontweight="bold")
ax1.set_ylabel("People", fontsize=11)
for bar, val in zip(bars1, pop_vals):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
             f"{val:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

# Income comparison
inc_vals = [mode_data[m]["median_income"] for m in modes]
bars2 = ax2.bar(modes, inc_vals, color=colors, edgecolor="white", width=0.5)
ax2.set_title("Mean Median Income\n(Raleigh, 15 min)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Dollars ($)", fontsize=11)
for bar, val in zip(bars2, inc_vals):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
             f"${val:,.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

fig.suptitle("Drive vs. Walk: Same City, Different Reach",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

The difference is dramatic. A 15-minute drive typically covers tens of thousands of
people across dozens of block groups. A 15-minute walk covers only the immediate
neighborhood — often just a handful of block groups with a fraction of the population.

The income comparison is also telling: the walkable area around a city center may have
a different income profile than the broader driving radius, reflecting the economic
character of the downtown core versus the suburbs.

---

## 7. Richer Variable Set and Horizontal Bar Chart

Let us expand the analysis to include five variables and visualize all of them in a
horizontal bar chart that makes it easy to compare cities across multiple dimensions
at a glance.

In [ ]:
rich_results = analyze_multiple_pois(
    locations=cities,
    travel_time=15,
    variables=["population", "median_income", "median_age", "poverty", "housing_units"],
)

# Quick summary of comparison winners
print("Comparison highlights:")
print(f"{'Variable':<18} {'Highest':<22} {'Lowest':<22}")
print(f"{'-'*18} {'-'*22} {'-'*22}")
for var in ["population", "median_income", "median_age", "poverty", "housing_units"]:
    info = rich_results["comparison"][var]
    print(f"{var:<18} {info['highest']:<22} {info['lowest']:<22}")

In [ ]:
# Build a DataFrame for plotting — normalize each variable to percentage of max
# so that all variables are on a comparable 0-100 scale.
variables_to_plot = ["population", "median_income", "median_age", "poverty", "housing_units"]
display_labels = {
    "population": "Population (total)",
    "median_income": "Median Income (mean)",
    "median_age": "Median Age (mean)",
    "poverty": "Poverty (total)",
    "housing_units": "Housing Units (total)",
}

# Use 'total' for count-like variables, 'mean' for rate-like variables
stat_for_var = {
    "population": "total",
    "median_income": "mean",
    "median_age": "mean",
    "poverty": "total",
    "housing_units": "total",
}

raw_data = {}
for loc in rich_results["locations"]:
    city_label = loc["location"]
    raw_data[city_label] = {}
    for var in variables_to_plot:
        stat_key = stat_for_var[var]
        raw_data[city_label][var] = loc["aggregated"].get(var, {}).get(stat_key, 0)

df_raw = pd.DataFrame(raw_data).T

# Normalize to percentage of column max for visual comparison
df_norm = df_raw.div(df_raw.max(axis=0), axis=1) * 100

# Plot horizontal grouped bar chart
fig, ax = plt.subplots(figsize=(10, 6))

y_positions = np.arange(len(variables_to_plot))
n_cities = len(df_norm)
bar_height = 0.22
city_colors = ["#4C78A8", "#F58518", "#E45756"]

for i, (city_label, row) in enumerate(df_norm.iterrows()):
    offset = (i - n_cities / 2 + 0.5) * bar_height
    values = [row[var] for var in variables_to_plot]
    bars = ax.barh(y_positions + offset, values, bar_height,
                   label=city_label, color=city_colors[i], edgecolor="white")

    # Add raw value labels
    for j, (bar, var) in enumerate(zip(bars, variables_to_plot)):
        raw_val = df_raw.loc[city_label, var]
        if stat_for_var[var] == "mean":
            label_text = f"${raw_val:,.0f}" if var == "median_income" else f"{raw_val:,.1f}"
        else:
            label_text = f"{raw_val:,.0f}"
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                label_text, va="center", fontsize=8)

ax.set_yticks(y_positions)
ax.set_yticklabels([display_labels[v] for v in variables_to_plot], fontsize=10)
ax.set_xlabel("% of Highest City", fontsize=11)
ax.set_title("Multi-Variable City Comparison (15-min Drive)",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(loc="lower right", framealpha=0.9, fontsize=10)
ax.set_xlim(0, 115)  # room for labels

fig.tight_layout()
plt.show()

This chart normalizes each variable to a percentage of the city with the highest value,
making it possible to compare across variables that have very different units and scales.
The raw values are printed next to each bar so you do not lose the actual numbers.

Reading the chart:
- A bar reaching 100% means that city has the highest value for that variable.
- Shorter bars show how much lower the other cities are, proportionally.
- If all three bars are roughly equal for a variable, the cities are similar on that
  dimension.

---

## 8. Generate an HTML Report

`generate_report` takes the analysis data dictionary and renders it as a styled HTML
document. This is useful for sharing results with colleagues, embedding in a web page,
or saving to disk.

The report includes:
- **Analysis parameters** — travel time, mode, variables requested
- **Per-location tables** — showing total and mean for each variable
- **Comparison section** — identifying which location ranks highest and lowest

The function signature is:

```python
generate_report(analysis_data, format="html") -> str
```

where `analysis_data` is the dict returned by `analyze_multiple_pois`.

In [ ]:
report_html = generate_report(rich_results, format="html")

print(f"Report length: {len(report_html):,} characters")
print(f"Report type:   {type(report_html).__name__}")
print(f"\nFirst 200 characters:")
print(report_html[:200])

In [ ]:
# Render the report inline in the notebook
display(HTML(report_html))

The rendered HTML report provides a self-contained summary of the entire analysis.
You can save it to a file with standard Python:

```python
with open("report.html", "w") as f:
    f.write(report_html)
```

---

## 9. Coordinate Tuple Input

So far we have used city name strings like `"Raleigh, NC"`. You can also pass
`(latitude, longitude)` tuples — or mix both formats in the same list.

This is useful when you have specific coordinates from a GPS, a geocoding service, or
a database of site locations. The function geocodes strings and passes tuples straight
through to the isochrone engine.

In [ ]:
mixed = analyze_multiple_pois(
    locations=[
        "Raleigh, NC",               # string — will be geocoded
        (35.2271, -80.8431),          # Charlotte coordinates — used directly
    ],
    travel_time=10,
    variables=["population"],
)

for loc in mixed["locations"]:
    pop = loc["aggregated"]["population"]["total"]
    bgs = loc["block_group_count"]
    print(f"{loc['location']}: {pop:,.0f} people in {bgs} block groups")

When a coordinate tuple is used, the `location` field in the results will show the
formatted coordinates (e.g., `35.2271, -80.8431`) rather than a city name. All other
fields work identically.

---

## Summary

This notebook demonstrated how to compare multiple locations using SocialMapper's
comparative analysis pipeline. Here are the key concepts:

| Concept | Details |
|---|---|
| Comparative analysis | Reveals spatial inequities, informs resource allocation, supports policy decisions |
| Parallel execution | `analyze_multiple_pois` runs locations concurrently (up to 4 threads) |
| Aggregated stats | `total`, `mean`, `min`, `max`, `count` per variable per location |
| Comparison rankings | Sorted by `total`, with `highest` and `lowest` shortcuts |
| Travel mode impact | Walk isochrones cover far less area than drive isochrones |
| HTML reports | `generate_report()` renders a self-contained HTML document |
| Mixed inputs | Strings and `(lat, lon)` tuples can be combined in one call |

### API Quick Reference

```python
# Analyze and compare
results = analyze_multiple_pois(
    locations=["City A", "City B", (lat, lon)],
    travel_time=15,          # minutes
    travel_mode="drive",     # "drive", "walk", or "bike"
    variables=["population", "median_income"],
    compare=True,            # set False to skip ranking
)

# Access results
results["locations"][0]["aggregated"]["population"]["total"]
results["comparison"]["population"]["highest"]

# Generate report
html = generate_report(results, format="html")
```

**Next notebook:** [07 — Complete Analysis Workflow](07-complete-analysis-workflow.ipynb)